In [ ]:
!pip install -q py-feat==2.1.3
!apt-get -qq install -y ffmpeg

In [ ]:
import os, time, subprocess, datetime
import torch, scipy.integrate, feat
from google.colab import drive

if not hasattr(scipy.integrate, "simps"):
    scipy.integrate.simps = scipy.integrate.simpson

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/ThesisExperiment_cleann'
OUT_DIR = os.path.join(BASE_DIR, 'detector_comparison')
os.makedirs(OUT_DIR, exist_ok=True)

SOURCE_VIDEO = os.path.join(BASE_DIR, 'participant_1_Robot1_cam.avi')
assert os.path.exists(SOURCE_VIDEO), f"Video not found: {SOURCE_VIDEO}"
TEMP_MP4 = '/content/temp_comparison_test.mp4'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FRAME_SAMPLE_INTERVAL = 5   # same as the main analysis

torch.backends.cudnn.enabled = False   # avoids the T4 "unable to find an engine" error
print("py-feat version:", feat.__version__, "| device:", DEVICE)

# same preprocessing as the real pipeline
subprocess.run(["ffmpeg", "-y", "-i", SOURCE_VIDEO, "-vf", "scale=640:-1",
                "-c:v", "libx264", "-preset", "ultrafast", "-g", "30",
                "-movflags", "+faststart", TEMP_MP4], check=True)

# ---- Detectorv1 ----
try:
    from feat import Detector as DetectorV1
except ImportError:
    from feat import Detectorv1 as DetectorV1
import inspect
params = inspect.signature(DetectorV1.__init__).parameters
wanted = dict(face_model="retinaface", landmark_model="mobilefacenet",
              au_model="xgb", emotion_model="svm", device=DEVICE, n_jobs=1)
kwargs = {k: v for k, v in wanted.items() if k in params}
dropped = [k for k in wanted if k not in params]
print("Detectorv1 settings used:", kwargs)
print("Detectorv1 settings not supported in this version:", dropped)
detector_v1 = DetectorV1(**kwargs)
t0 = time.time()
if hasattr(detector_v1, "detect_video"):
    pred1 = detector_v1.detect_video(TEMP_MP4, batch_size=8,
                                     every_n_frame=FRAME_SAMPLE_INTERVAL, num_workers=0)
else:
    pred1 = detector_v1.detect(TEMP_MP4, data_type="video",
                               skip_frames=FRAME_SAMPLE_INTERVAL, num_workers=0)
t1 = time.time() - t0
pred1.to_csv(os.path.join(OUT_DIR, 'comparison_v1_output.csv'), index=False)
print(f"v1: {t1:.1f}s, {len(pred1)} frames")

# ---- Detectorv2 ----
from feat import Detectorv2
detector_v2 = Detectorv2(device=DEVICE)
t0 = time.time()
pred2 = detector_v2.detect(TEMP_MP4, data_type="video",
                           skip_frames=FRAME_SAMPLE_INTERVAL, num_workers=0)
t2 = time.time() - t0
pred2.to_csv(os.path.join(OUT_DIR, 'comparison_v2_output.csv'), index=False)
print(f"v2: {t2:.1f}s, {len(pred2)} frames")

# ---- run info for the thesis record ----
with open(os.path.join(OUT_DIR, 'comparison_run_info.txt'), 'w') as f:
    f.write(f"Run on: {datetime.datetime.now():%Y-%m-%d %H:%M}\n")
    f.write(f"py-feat version: {feat.__version__}\nDevice: {DEVICE}\n")
    f.write(f"Source video: {os.path.basename(SOURCE_VIDEO)}\n")
    f.write(f"Frame sample interval: {FRAME_SAMPLE_INTERVAL}\n")
    f.write(f"Detectorv1: {t1:.1f}s, {len(pred1)} frames\n")
    f.write(f"Detectorv2: {t2:.1f}s, {len(pred2)} frames\n")
os.remove(TEMP_MP4)
print("Saved to:", OUT_DIR, os.listdir(OUT_DIR))